# Notebook 02 — Risk Adjustment

**Purpose:** Apply two haircuts to the expected return vector from Notebook 01. Account for smart contract exploit risk and liquidity cost.

**Inputs:** 
- `data/processed/expected_returns.csv`
- `data/processed/protocol_params.json`

**Outputs:** 
- `data/processed/risk_adjusted_returns.csv`

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import json

# Add src to path
sys.path.append(os.path.abspath('../src'))

import risk_adjustment as risk
import utils

np.random.seed(42)

## 1. Load Data and Params

In [ ]:
master_df = pd.read_csv('../data/processed/expected_returns.csv')
params = utils.load_protocol_params('../data/processed/protocol_params.json')

summary = master_df.groupby('instrument')['yield_idr'].mean().reset_index()
summary

## 2. Apply Haircuts

We apply:
1. **Expected Loss Haircut**: `p * s`
2. **Liquidity Haircut**: basis points penalty based on liquidity score.

In [ ]:
adjusted_rows = []
for inst in summary['instrument']:
    p_s = params[inst]
    mean_y = summary.loc[summary['instrument'] == inst, 'yield_idr'].iloc[0]
    
    # Haircut 1: Exploit
    y_exploit = risk.apply_exploit_haircut(mean_y, p_s['p'], p_s['s'])
    
    # Haircut 2: Liquidity
    y_final = risk.apply_liquidity_haircut(y_exploit, p_s['liquidity_score'])
    
    adjusted_rows.append({
        'instrument': inst,
        'mean_yield_idr': mean_y,
        'expected_loss_haircut': p_s['p'] * p_s['s'],
        'liquidity_haircut': mean_y - y_exploit + (y_exploit - y_final), # wait, this is just total diff
        'risk_adjusted_yield_idr': y_final
    })
    
risk_df = pd.DataFrame(adjusted_rows)
# Calculate liquidity_haircut specifically
risk_df['liquidity_haircut'] = risk_df['mean_yield_idr'] - risk_df['expected_loss_haircut'] - risk_df['risk_adjusted_yield_idr']

risk_df.to_csv('../data/processed/risk_adjusted_returns.csv', index=False)
risk_df